In [ ]:
"""
Objective: Estimate univariate Persistence/Half-life for each credit portfolio and apply 
Jordà's (2005) Local Projections method to map impacts on unemployment.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.ar_model import ar_select_order, AutoReg
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. DATA LOADING AND MAPPING
# ==========================================
def load_lp_data(csv_file: str) -> pd.DataFrame:
    """Loads the dataset preserving the maximum available sample (since 2000)."""
    print("[*] Loading clean dataset...")
    df = pd.read_csv(csv_file, index_col=0, parse_dates=True)
    df_model = df.loc['2000-01-01':'2026-04-01'].copy()
    
    print(f"[*] Extended sample: {df_model.index[0].date()} to {df_model.index[-1].date()}")
    print(f"[*] Total observations: {len(df_model)} quarters")
    return df_model

PORTFOLIOS = {
    'delinq_consumer': 'Consumer Credit',
    'delinq_mortgages': 'Mortgages',
    'delinq_credit_cards': 'Credit Cards'
}

# ==========================================
# 2. PERSISTENCE AND HALF-LIFE (UNIVARIATE AR)
# ==========================================
def calculate_half_life(df: pd.DataFrame, portfolio_dict: dict):
    """Calculates persistence and half-life using an AR(p) model selected via BIC."""
    print("\n--- Persistence and Half-Life (Internal Shock) ---")
    for col, name in portfolio_dict.items():
        series = df[col].dropna()
        optimal_lags = ar_select_order(series, maxlag=4, ic='bic').ar_lags
        ar_model = AutoReg(series, lags=optimal_lags).fit()
        
        ar_sum = np.sum(ar_model.params[1:])
        if 0 < ar_sum < 1:
            half_life = np.log(0.5) / np.log(ar_sum)
            print(f"[{name}] Order: AR({len(optimal_lags)}) | Persistence: {ar_sum:.3f} | Half-life: {half_life:.2f} qtrs.")
        else:
            print(f"[{name}] Order: AR({len(optimal_lags)}) | Series shows unit root or no mean-reversion.")

# ==========================================
# 3. JORDÀ LOCAL PROJECTIONS (ESTIMATION & CHARTS)
# ==========================================
def run_local_projections(df: pd.DataFrame, portfolio_dict: dict, h_max: int = 12):
    """
    Estimates Local Projections vectors using HAC (Newey-West) regressions 
    and exports Impulse Response Function (IRF) charts.
    """
    print("\n[*] Estimating Local Projections (Horizon: 12 quarters)...")
    
    # Response variable (Unemployment) in first difference
    y = df['unemployment'].diff().dropna()
    
    for col, name in portfolio_dict.items():
        # Contemporaneous shock also in difference
        x = df[col].diff().dropna()
        
        betas, ic_lower, ic_upper = [], [], []
        
        for h in range(h_max + 1):
            y_future = y if h == 0 else y.shift(-h)
            
            data = pd.DataFrame({
                'y_future': y_future, 'x_shock': x,
                'y_lag1': y.shift(1), 'x_lag1': x.shift(1)
            }).dropna()
            
            X = sm.add_constant(data[['x_shock', 'y_lag1', 'x_lag1']])
            Y = data['y_future']
            
            hac_result = sm.OLS(Y, X).fit(cov_type='HAC', cov_kwds={'maxlags': h + 1})
            
            betas.append(hac_result.params['x_shock'])
            ic_lower.append(hac_result.conf_int(alpha=0.05).loc['x_shock', 0])
            ic_upper.append(hac_result.conf_int(alpha=0.05).loc['x_shock', 1])
            
        # Export chart
        _plot_lp_irf(name, list(range(h_max + 1)), betas, ic_lower, ic_upper, h_max)

def _plot_lp_irf(name: str, horizon: list, irf: list, ic_lower: list, ic_upper: list, h_max: int):
    """Helper function to save Local Projection charts."""
    fig, ax = plt.subplots(figsize=(8, 5))
    
    ax.plot(horizon, irf, color='firebrick', linewidth=2, label='Estimated Response (LP)')
    ax.fill_between(horizon, ic_lower, ic_upper, color='red', alpha=0.15, label='95% Confidence Interval')
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    
    ax.set_ylabel('Impact on Unemployment (p.p.)', fontsize=10, fontweight='bold')
    ax.set_xlabel('Horizon (Quarters)', fontsize=10, fontweight='bold')
    ax.legend(loc='best', fontsize=9, frameon=False)
    ax.grid(True, linestyle='--', alpha=0.3)
    ax.set_xticks(range(0, h_max + 1, 2))
    plt.tight_layout()
    
    file_name = name.lower().replace(' ', '_')
    output_file = f'02_irf_lp_{file_name}.png'
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"[*] Chart exported: '{output_file}'")

if __name__ == "__main__":
    df_model = load_lp_data('clean_macro_fred_dataset.csv')
    calculate_half_life(df_model, PORTFOLIOS)
    run_local_projections(df_model, PORTFOLIOS, h_max=12)